|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>The block allocator<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: build the allocator and the page table<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

Build the allocator.

No tensors and no GPU. This is stage 06 of the ladder, and it is pure
bookkeeping, which is exactly why it is worth getting right on its own before
any kernel has to read through it.

In [ ]:
### run this cell

BLOCK    = 16                # tokens per block
N_BLOCKS = 4096              # blocks in the pool
lengths  = rng.lognormal(mean=np.log(120), sigma=0.9, size=2000).astype(int) + 1

print(f'pool holds {N_BLOCKS*BLOCK:,} tokens in {N_BLOCKS:,} blocks of {BLOCK}')

# Exercise 1: the free list

A pool of blocks, a stack of the free ones, and a reference count per block.
The refcount looks unnecessary now. Exercise 4 is why it is there.

In [ ]:
class BlockAllocator:
  def __init__(self, n_blocks):
    self.free = list(range(n_blocks))      # a stack of physical ids
    self.ref  = [0] * n_blocks

  def allocate(self):
    if not self.free:
      raise MemoryError('out of KV blocks')
    b = self.free.pop()
    self.ref[b] = 1
    return b

  def release(self, b):
    self.ref[b] -= 1
    if self.ref[b] == 0:
      self.free.append(b)

  def share(self, b):
    self.ref[b] += 1
    return b

  @property
  def used(self):
    return len(self.ref) - len(self.free)

a = BlockAllocator(8)
x, y = a.allocate(), a.allocate()
print('used after 2 allocations:', a.used)
a.release(x)
print('used after 1 release:    ', a.used)

# Exercise 2: the block table

One per sequence. It maps a logical token position onto a physical slot in
the pool, and it grows one block at a time as the sequence does.

In [ ]:
class BlockTable:
  """One sequence's view of the pool."""
  def __init__(self, allocator, block_size):
    self.alloc  = allocator
    self.bs     = block_size
    self.blocks = []      # logical block index -> physical block id
    self.n      = 0       # tokens held

  def append_token(self):
    if self.n % self.bs == 0:            # the current block is full
      self.blocks.append(self.alloc.allocate())
    self.n += 1

  def slot_index(self, pos):
    """logical token position -> flat slot in the pool"""
    return self.blocks[pos // self.bs] * self.bs + pos % self.bs

  def free(self):
    for b in self.blocks:
      self.alloc.release(b)
    self.blocks, self.n = [], 0

alloc = BlockAllocator(N_BLOCKS)
t = BlockTable(alloc, BLOCK)
for _ in range(40):
  t.append_token()
print(f'40 tokens -> {len(t.blocks)} blocks: {t.blocks}')
print(f'position 0  -> slot {t.slot_index(0)}')
print(f'position 17 -> slot {t.slot_index(17)}')

# Exercise 3: how many sequences fit now?

Admit requests from the workload until the allocator refuses, and compare
with reserving `max_len` for each of them.

In [ ]:
alloc  = BlockAllocator(N_BLOCKS)
tables = []
admitted = 0

for L in lengths:
  t = BlockTable(alloc, BLOCK)
  try:
    for _ in range(int(L)):
      t.append_token()
  except MemoryError:
    t.free()
    break
  tables.append(t); admitted += 1

held  = sum(len(t.blocks) for t in tables) * BLOCK
used  = sum(t.n for t in tables)
print(f'admitted {admitted} sequences before the pool ran out')
print(f'tokens held {held:,}, tokens used {used:,}  -> {100*(1-used/held):.1f}% wasted')

MAX_LEN = 2048
contig  = (N_BLOCKS*BLOCK) // MAX_LEN
print(f'\ncontiguous, reserving {MAX_LEN}: {contig} sequences')
print(f'paged:                     {admitted} sequences   ({admitted/contig:.0f}x)')

# Exercise 4: two sequences, one prompt

Parallel sampling asks the model for four replies to the same prompt. The
prompt's K and V are identical for all four.

With a page table, that costs nothing extra.

In [ ]:
alloc = BlockAllocator(N_BLOCKS)
before = alloc.used

parent = BlockTable(alloc, BLOCK)
for _ in range(64):
  parent.append_token()
after_parent = alloc.used

# four samples from the same prompt: share every block the prompt holds
children = []
for _ in range(4):
  c = BlockTable(alloc, BLOCK)
  c.blocks = [alloc.share(b) for b in parent.blocks]
  c.n = parent.n
  children.append(c)

print(f'prompt of 64 tokens costs      {after_parent - before} blocks')
print(f'4 samples sharing it cost      {alloc.used - after_parent} more')
print(f'4 samples copying it would be  {4*(after_parent-before)} more')

for c in children:
  c.free()
print(f'\nafter the children leave, still held: {alloc.used} blocks (the parent)')

### What you built

An allocator, a page table, and a refcount. About sixty lines, no
tensors, and it is worth roughly ten times the memory efficiency of the
thing it replaced.

Exercise 4 is the part to remember. Sharing a prefix became **a pointer
operation**. Four samples from one prompt cost four block-table entries
instead of four copies of the prompt's KV cache, and the refcount means
nobody frees a block another sequence is still reading.

Follow that thread and you get the rest of stage 09:

- one sequence writes into a shared block, so copy it first, for that
  sequence only. Copy-on-write, exactly as `fork` does it.
- hash a block's contents and two unrelated **requests** with the same
  system prompt share it. Automatic prefix caching, which is a page
  cache.

None of this was available while the cache was one contiguous buffer.
It is not that paging made sharing faster; paging made sharing
expressible.

    ./vc guide 6